# 🚁 Drone Thermal Anomaly Detection - Colab Training
This notebook trains the YOLOv8 model for the drone thermal anomaly detection system. It will:
1. Setup the environment
2. Download the FLIR dataset using Kaggle
3. Preprocess it into YOLO format
4. Train YOLOv8 on the dataset
5. Export the weights so you can use them locally.

In [ ]:
!pip install ultralytics opencv-python

## 1. Download Dataset (Kaggle)
Enter your Kaggle credentials below. You can find them in `kaggle.json` created from your Kaggle Account settings.

In [ ]:
import os
import getpass

os.environ['KAGGLE_USERNAME'] = input('Kaggle Username: ')
os.environ['KAGGLE_KEY'] = getpass.getpass('Kaggle API Key: ')

!mkdir -p raw_flir
!echo "Downloading dataset (this may take a few minutes)..."
!curl -L -u "{os.environ['KAGGLE_USERNAME']}:{os.environ['KAGGLE_KEY']}" -o raw_flir/dataset.zip "https://www.kaggle.com/api/v1/datasets/download/deepnewbie/flir-thermal-images-dataset"

!echo "Unzipping..."
!cd raw_flir && unzip -q dataset.zip && rm dataset.zip
!echo "Download complete!"

## 2. Prepare Dataset
Converting the COCO JSON annotations to YOLO format and taking a subset of the data.

In [ ]:
import json, shutil, random, os
from pathlib import Path

RAW_DATA_DIR = Path("raw_flir")
PROCESSED_DIR = Path("processed")
YOLO_CLASSES = {"person": 0, "car": 1, "bicycle": 2}

for split in ["train", "val"]:
    os.makedirs(PROCESSED_DIR / split / "images", exist_ok=True)
    os.makedirs(PROCESSED_DIR / split / "labels", exist_ok=True)

def convert_coco_to_yolo(coco_json_path, image_dir, split_name, subset_fraction=1.0):
    with open(coco_json_path, 'r') as f: coco_data = json.load(f)
    
    category_map = {cat['id']: YOLO_CLASSES[cat['name']] for cat in coco_data['categories'] if cat['name'] in YOLO_CLASSES}
    images = random.sample(coco_data['images'], int(len(coco_data['images']) * subset_fraction))
    print(f"[{split_name}] Processing {len(images)} images ({subset_fraction*100}% subset)...")
    
    image_map = {img['id']: img for img in images}
    annotations_by_image = {img_id: [] for img_id in image_map.keys()}
    
    for ann in coco_data['annotations']:
        img_id = ann['image_id']
        if img_id in image_map and ann['category_id'] in category_map:
            img = image_map[img_id]
            x, y, w, h = ann['bbox']
            x_center = (x + w / 2) / img['width']
            y_center = (y + h / 2) / img['height']
            norm_w = w / img['width']
            norm_h = h / img['height']
            annotations_by_image[img_id].append(f"{category_map[ann['category_id']]} {x_center} {y_center} {norm_w} {norm_h}")

    for img_id, img_info in image_map.items():
        src_img_path = image_dir / os.path.basename(img_info['file_name'])
        if not src_img_path.exists(): continue
        base_name = os.path.splitext(os.path.basename(img_info['file_name']))[0]
        shutil.copy2(src_img_path, PROCESSED_DIR / split_name / "images" / f"{base_name}.jpg")
        with open(PROCESSED_DIR / split_name / "labels" / f"{base_name}.txt", 'w') as f:
            if annotations_by_image[img_id]: f.write('\n'.join(annotations_by_image[img_id]) + '\n')

def find_json_and_images(split):
    for p in RAW_DATA_DIR.rglob('*.json'):
        if split in p.parts and 'thermal_annotations' in p.name:
            img_dir = p.parent / 'thermal_8_bit'
            if not img_dir.exists():
                img_dir = p.parent
            return p, img_dir
    return None, None

train_json, train_imgs = find_json_and_images('train')
if train_json:
    convert_coco_to_yolo(train_json, train_imgs, 'train', 1.0)
else:
    print(f'Error: Could not find train annotations inside {RAW_DATA_DIR}')

val_json, val_imgs = find_json_and_images('val')
if val_json:
    convert_coco_to_yolo(val_json, val_imgs, 'val', 1.0)

yaml_content = f"""path: {PROCESSED_DIR.absolute()}
train: train/images
val: val/images
nc: 3
names: ['person', 'car', 'bicycle']
"""
with open("flir.yaml", "w") as f: f.write(yaml_content)
print("Dataset ready!")

## 3. Train YOLOv8 Model
Now we fine-tune a YOLOv8 nano model on our thermal dataset.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Train the model for 20 epochs (increase for better accuracy if you have time)
results = model.train(data='flir.yaml', epochs=20, imgsz=640, workers=2, batch=16, project='aerial-thermal', name='yolov8_run1')

## 4. Download Weights
Download the `best.pt` file to your local machine so you can use it in your backend API.

In [ ]:
from google.colab import files

weights_path = 'aerial-thermal/yolov8_run1/weights/best.pt'
if os.path.exists(weights_path):
    print("Downloading weights to your local machine...")
    files.download(weights_path)
    print("\nPlease place this file in your local repository at: ai-model/weights/best.pt")
else:
    print("Weights not found! Check if the training completed successfully.")